# Health checks

Quick checks on a Weaviate instance (open source, DigitalOcean, Weaviate Cloud, ...):

- **Collection count**: many collections often mean multi-tenancy would fit better.
- **Replication**: replication factor against the node count, async replication and the deletion strategy.
- **Similar configurations**: collections with identical settings are candidates for one multi-tenant collection.
- **Compression**: vectors without quantization use more memory.

Connection settings are read from a `.env` file next to the notebook (or from environment variables). The connection cell lists them.

In [ ]:
%pip install -U weaviate-client python-dotenv

In [ ]:
import os

import httpx
import weaviate
from dotenv import load_dotenv
from weaviate.classes.init import AdditionalConfig, Auth, Timeout

# Settings come from a .env file (or environment variables):
#   WEAVIATE_HOST       host without http:// or https://   (default: localhost)
#   WEAVIATE_HTTP_PORT  default 8080                       (443 for Weaviate Cloud)
#   WEAVIATE_GRPC_HOST  default: same as WEAVIATE_HOST     (grpc-<host> for Weaviate Cloud)
#   WEAVIATE_GRPC_PORT  default 50051                      (443 for Weaviate Cloud)
#   WEAVIATE_SECURE     true for HTTPS/TLS                 (true for Weaviate Cloud)
#   WEAVIATE_API_KEY    leave unset for anonymous access
# Weaviate Cloud can also use weaviate.connect_to_weaviate_cloud(cluster_url=..., auth_credentials=Auth.api_key(...)).
load_dotenv(".env", override=True)  # .env next to the notebook

HOST = os.getenv("WEAVIATE_HOST", "localhost")
HTTP_PORT = int(os.getenv("WEAVIATE_HTTP_PORT", "8080"))
GRPC_HOST = os.getenv("WEAVIATE_GRPC_HOST") or HOST
GRPC_PORT = int(os.getenv("WEAVIATE_GRPC_PORT", "50051"))
SECURE = os.getenv("WEAVIATE_SECURE", "false").lower() == "true"
API_KEY = os.getenv("WEAVIATE_API_KEY")

client = weaviate.connect_to_custom(
    http_host=HOST,
    http_port=HTTP_PORT,
    http_secure=SECURE,
    grpc_host=GRPC_HOST,
    grpc_port=GRPC_PORT,
    grpc_secure=SECURE,
    auth_credentials=Auth.api_key(API_KEY) if API_KEY else None,
    additional_config=AdditionalConfig(timeout=Timeout(init=30, query=60, insert=120)),
)

# The raw schema, exactly as the server reports it
response = httpx.get(
    f"{'https' if SECURE else 'http'}://{HOST}:{HTTP_PORT}/v1/schema",
    headers={"Authorization": f"Bearer {API_KEY}"} if API_KEY else {},
    timeout=30,
)
response.raise_for_status()
SCHEMA = {c["class"]: c for c in response.json().get("classes") or []}

print(f"Weaviate {client.get_meta()['version']} (client {weaviate.__version__})")
print(f"Ready: {client.is_ready()} | Live: {client.is_live()} | Connected: {client.is_connected()}")
print(f"Collections: {len(SCHEMA)}")

In [ ]:
# --- Collection count ---
print(f"Number of collections: {len(SCHEMA)}")

# Many collections often means one per user or group, which multi-tenancy handles better.
if len(SCHEMA) > 100:
    print("Warning: you have a lot of collections.")
    print("Consider multi-tenancy (one multi-tenant collection) for better scalability,")
    print("especially if many collections are configured identically but represent different users, groups etc.")
else:
    print("You have a reasonable number of collections.")

In [ ]:
# --- Replication ---
RECOMMENDED_DELETION = {"TimeBasedResolution", "DeleteOnConflict"}
nodes = len(client.cluster.nodes())
print(f"Nodes: {nodes}\n")

for name, cls in sorted(SCHEMA.items()):
    replication = cls.get("replicationConfig") or {}
    factor = replication.get("factor", 1)
    async_enabled = replication.get("asyncEnabled")
    deletion = replication.get("deletionStrategy")

    notes = []
    if factor > nodes:
        notes.append(f"factor {factor} is higher than the node count ({nodes})")
    if factor == 1 and nodes > 1:
        notes.append("no replicas: a node outage makes this collection unavailable; 3 is common for production")
    if factor > 1 and async_enabled is not True:
        notes.append("enable async replication so replicas stay in sync in the background")
    if factor > 1 and deletion not in RECOMMENDED_DELETION:
        notes.append(f"deletion strategy is {deletion}; set TimeBasedResolution or DeleteOnConflict")

    print(f"- {name}: factor={factor} | async={async_enabled} | deletion={deletion}")
    for note in notes:
        print(f"    -> {note}")

print()
print("The replication factor does not need to match the node count: 3 replicas on a 5 or 9 node cluster is normal.")
print("Deletion strategies when replicas disagree:")
print("- TimeBasedResolution: the most recent change wins (delete, create or update).")
print("- DeleteOnConflict: a delete on any replica deletes the object everywhere.")

In [ ]:
# --- Similar configurations ---
import json


def core_settings(cls):
    """The settings that decide whether collections could share one multi-tenant collection."""
    vectors = {}
    for name, cfg in sorted((cls.get("vectorConfig") or {}).items()):
        vectorizer = cfg.get("vectorizer") or {}
        provider = next(iter(vectorizer), None)
        details = vectorizer.get(provider) or {}
        vectors[name] = {
            "provider": provider,
            "model": details.get("model"),
            "properties": details.get("properties"),
            "vectorIndexType": cfg.get("vectorIndexType"),
            "vectorIndexConfig": cfg.get("vectorIndexConfig"),
        }
    settings = {
        "replicationConfig": cls.get("replicationConfig"),
        "invertedIndexConfig": cls.get("invertedIndexConfig"),
        "vectorConfig": vectors,
        "vectorizer": cls.get("vectorizer"),  # collections without named vectors
        "vectorIndexConfig": cls.get("vectorIndexConfig"),
        "properties": sorted((p["name"], tuple(p["dataType"])) for p in cls.get("properties") or []),
    }
    return json.dumps(settings, sort_keys=True)


groups = {}
for name, cls in SCHEMA.items():
    if not (cls.get("multiTenancyConfig") or {}).get("enabled"):
        groups.setdefault(core_settings(cls), []).append(name)

candidates = [sorted(names) for names in groups.values() if len(names) > 1]
if not candidates:
    print("No single-tenant collections share identical settings.")
for names in candidates:
    print(f"{len(names)} collections share identical settings and properties, candidates for one multi-tenant collection:")
    print("  " + ", ".join(names))

In [ ]:
# --- Compression ---
QUANTIZERS = ("pq", "bq", "sq", "rq")


def quantizers(index_config):
    """Enabled quantizers in a vector index config, including both halves of a dynamic index."""
    index_config = index_config or {}
    found = {q for q in QUANTIZERS if (index_config.get(q) or {}).get("enabled")}
    for nested in ("hnsw", "flat"):  # dynamic index
        if isinstance(index_config.get(nested), dict):
            found |= quantizers(index_config[nested])
    return found


for name, cls in sorted(SCHEMA.items()):
    if cls.get("vectorConfig"):
        vectors = {v: cfg.get("vectorIndexConfig") for v, cfg in cls["vectorConfig"].items() if cfg.get("vectorIndexType") != "none"}
    else:
        vectors = {"default": cls.get("vectorIndexConfig")}  # collections without named vectors
    for vector, index_config in sorted(vectors.items()):
        found = quantizers(index_config)
        status = ", ".join(sorted(found)) if found else "none -> consider RQ (or BQ/SQ/PQ) to reduce memory"
        print(f"- {name} / {vector}: compression {status}")

In [ ]:
client.close()